In [4]:
import pandas as pd 
import numpy as np 
import re

In [2]:
raw=pd.read_excel("project raw file.xlsx")
raw

,stNo,EmpName,Unnamed: 2
0,ST2462,Theodora Kansiime,NaN
1,ST1917,Oola Hilda,NaN
2,ST2208,Esebu Edward,NaN
3,ST2735,Muwumba Norah,NaN
4,ST1787,Kababiito Winfred,NaN
...,...,...,...
340,ST2545,Shallot Atukunda,NaN
341,ST2419,Ileka Grace,NaN
342,ST2446,Emuria Joseph,NaN
343,ST1853,Mugisa Michael,NaN


In [ ]:
# Standardize column names
raw.columns = [str(c).strip() for c in raw.columns]
id_col= 'stNo'
name_col='EmpName'




In [ ]:
# Build or update master reference from IDs
master = (
    raw[[id_col, name_col]]
    .dropna()
    .drop_duplicates(subset=[id_col], keep='first')
    .rename(columns={name_col: 'Standard_Name'})
)

master.to_excel(MASTER_FILE, index=False, engine='openpyxl')

# Apply standard names using ID
id_to_name = dict(zip(master[id_col], master['Standard_Name']))
raw['Standard_Name'] = raw[id_col].map(id_to_name)

# Money cleaning function (for future files)
def standardize_money(value):
    if pd.isna(value):
        return value

    text = str(value).lower().replace(',', '').strip()
    text = text.replace('ugx', '').strip()

    multipliers = {
        'k': 1_000,
        'thousand': 1_000,
        'm': 1_000_000,
        'million': 1_000_000,
        'b': 1_000_000_000,
        'billion': 1_000_000_000,
    }

    match = re.match(r'(\d+(?:\.\d+)?)\s*([a-z]+)?', text)
    if match:
        num = float(match.group(1))
        suffix = match.group(2)

        if suffix in multipliers:
            num *= multipliers[suffix]

        return int(num)

    return value

raw.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')

print(f'Created: {MASTER_FILE}')
print(f'Created: {OUTPUT_FILE}')
